## 環境設置

In [1]:
try:
    import openseespy.opensees as ops  # noqa: F401
except ImportError:
    import sys
    !{sys.executable} -m pip install -q openseespy
    import openseespy.opensees as ops  # noqa: F401

# Case-03.7:Demand 物件與 Design Loop

依討論後修正的順序,這一步在 Case-04(桃園案例)之前先把介面定案,
避免 Case-04 做完之後又要因為介面改版重做一次。

**這一步做兩件事**:
1. 把散裝的 `Pu, Mu, Vu` 參數包成統一的 `Demand` 物件,之後要加 θ、
   支承反力這些欄位,不需要每個檢核函式都改參數列
2. 把 Case-03.5/03.6 手動試 3 個尺寸的做法,改成 `while not ok:` 的
   自動收斂迴圈——這才是 ETABS/SAP2000 Auto Design 背後的核心概念,
   而且是可以公開重現的版本

**先聲明**:`interfaces/` 或 `src/taiwan_seismic_calc/` 的獨立檔案抽離
還沒有做——那個決定(要不要用 Colab 自動下載雙模式)刻意留到介面
穩定之後才處理,這裡的 `Demand`、`analyze_frame`、`design_loop` 仍然
寫在 notebook 內,是刻意的,不是忘記做。

## 第 1 課:Demand 物件

用 `dataclass` 定義統一的需求資料結構,取代散裝參數。

In [2]:
from dataclasses import dataclass, field

@dataclass
class Demand:
    Pu: float = 0.0                  # 軸力需求
    Mu: float = 0.0                  # 彎矩需求
    Vu: float = 0.0                  # 剪力需求
    displacement: float = None       # 側向位移
    drift: float = None              # 層間位移角(1F)
    reactions: dict = field(default_factory=dict)   # 其他附加資訊(如2F位移角)

print(Demand(Pu=100.0, Mu=20.0, Vu=5.0))

Demand(Pu=100.0, Mu=20.0, Vu=5.0, displacement=None, drift=None, reactions={})


## 第 2 課:分析函式改回傳 Demand 物件

沿用 Case-03/03.5/03.6 已經驗證過的剪力構架建模方法,唯一改變是
回傳值從裸資料改成 `Demand` 物件。

In [3]:
h1 = h2 = 3.5
L = 6.0
F1_frame, F2_frame = 9.938, 15.900   # 沿用Case-03.5的單榀構架分擔力
Pu_gravity = 147.60                    # 沿用Case-03.6的重力軸力概估
DRIFT_LIMIT = 0.005

def analyze_frame(h_col, E_mat):
    """沿用Case-03驗證過的剪力構架方法, 回傳Demand物件"""
    Ic = h_col**4/12
    ops.wipe()
    ops.model('basic', '-ndm', 2, '-ndf', 3)
    ops.node(1, 0.0, 0.0);   ops.node(2, L, 0.0)
    ops.node(3, 0.0, h1);    ops.node(4, L, h1)
    ops.node(5, 0.0, h1+h2); ops.node(6, L, h1+h2)
    ops.fix(1,1,1,1); ops.fix(2,1,1,1)
    for n in [3,4,5,6]:
        ops.fix(n, 0,1,1)
    ops.equalDOF(3,4,1); ops.equalDOF(5,6,1)
    A_big = 1.0e6
    ops.geomTransf('Linear', 1)
    ops.element('elasticBeamColumn',1,1,3,A_big,E_mat,Ic,1)
    ops.element('elasticBeamColumn',2,2,4,A_big,E_mat,Ic,1)
    ops.element('elasticBeamColumn',3,3,5,A_big,E_mat,Ic,1)
    ops.element('elasticBeamColumn',4,4,6,A_big,E_mat,Ic,1)
    ops.timeSeries('Linear',1); ops.pattern('Plain',1,1)
    ops.load(3, F1_frame, 0.0, 0.0)
    ops.load(5, F2_frame, 0.0, 0.0)
    ops.system('BandGeneral'); ops.numberer('RCM'); ops.constraints('Transformation')
    ops.test('NormDispIncr',1e-10,20); ops.algorithm('Newton')
    ops.integrator('LoadControl',1.0); ops.analysis('Static'); ops.analyze(1)

    u1 = ops.nodeDisp(3,1); u2 = ops.nodeDisp(5,1)
    f1 = ops.eleForce(1)
    return Demand(Pu=Pu_gravity, Mu=abs(f1[2]), Vu=abs(f1[1]),
                  displacement=u1, drift=u1/h1,
                  reactions={'drift2': (u2-u1)/h2})


E_rc = 2.463e7
demand_test = analyze_frame(0.40, E_rc)
print(demand_test)

# 迴歸測試: 確認重構後跟Case-03.6原始結果一致
assert abs(demand_test.Mu-22.608) < 0.01, "跟Case-03.6原始結果對不起來!"
print("\n[PASS] Demand物件重構後數字與Case-03.6原始結果一致")

Demand(Pu=147.6, Mu=22.60825, Vu=0.0, displacement=0.0008784734777836987, drift=0.00025099242222391394, reactions={'drift2': 0.00015445388626674782})

[PASS] Demand物件重構後數字與Case-03.6原始結果一致


## 第 3 課:檢核函式改吃 Demand 物件

同樣沿用 Case-03.6 驗證過的規範公式,只改參數介面。

In [4]:
def rc_check(h_col_m, demand):
    """RC方形柱快速強度+位移角檢核, 吃Demand物件"""
    fc, fy = 280.0, 4200.0
    rho = 0.02
    phi_axial = phi_moment = 0.65
    phi_shear = 0.75

    b = d = h_col_m*100
    Ag = b*d; Ast = rho*Ag
    Po = 0.85*fc*(Ag-Ast) + fy*Ast
    phiPn = phi_axial*0.80*Po*9.80665e-3
    a = Ast*fy/(0.85*fc*b)
    Mn = Ast*fy*(d-a/2)
    phiMn = phi_moment*Mn*9.80665e-5
    Vc = 0.53*(fc**0.5)*b*d
    phiVn = phi_shear*Vc*9.80665e-3

    drift_ok = (demand.drift < DRIFT_LIMIT) and (demand.reactions['drift2'] < DRIFT_LIMIT)
    m_util = demand.Mu/phiMn
    p_util = demand.Pu/phiPn
    v_util = demand.Vu/max(phiVn, 1e-9)
    strength_ok = m_util < 1.0 and p_util < 1.0 and v_util < 1.0

    return dict(pass_=(drift_ok and strength_ok), drift_ok=drift_ok, strength_ok=strength_ok,
                m_util=m_util, p_util=p_util, v_util=v_util,
                drift_util=demand.drift/DRIFT_LIMIT)


r = rc_check(0.40, demand_test)
print(r)
assert abs(r['m_util']-0.0801) < 0.001, "跟Case-03.6原始結果對不起來!"
print("[PASS] 檢核函式重構後數字與Case-03.6原始結果一致")

{'pass_': True, 'drift_ok': True, 'strength_ok': True, 'm_util': 0.08011144039135888, 'p_util': 0.057023571481439606, 'v_util': 0.0, 'drift_util': 0.05019848444478279}
[PASS] 檢核函式重構後數字與Case-03.6原始結果一致


## 第 4 課:Design Loop——`while not ok:` 的核心邏輯

這是整個 Case-03.7 真正的重點。輸入一組候選尺寸清單,自動依序試設、
分析、檢核,找到第一個通過的就停止——不用再像 Case-03.5 那樣手動
挑 3 個數字。

In [5]:
def design_loop(candidates, build_fn, check_fn, E_mat, verbose=True):
    """依序嘗試候選尺寸清單, 回傳第一個通過檢核的結果"""
    log = []
    for c in candidates:
        demand = build_fn(c, E_mat)
        result = check_fn(c, demand)
        log.append((c, demand, result))
        if verbose:
            print(f"  嘗試 {c}: drift_util={result['drift_util']:.1%}, "
                  f"M_util={result['m_util']:.1%}  -> {'PASS' if result['pass_'] else 'FAIL'}")
        if result['pass_']:
            return c, demand, result, log
    return None, None, None, log

## 第 5 課:RC 案例——Design Loop 自動收斂

跑一次跟 Case-03.6 完全相同的候選清單,驗證 Design Loop 自動找到的
答案是不是跟手動掃描一致(20cm)。

In [6]:
candidates_rc = [0.15, 0.18, 0.20, 0.25, 0.30, 0.35, 0.40]

print("=== RC Design Loop ===")
size_rc, demand_rc, result_rc, log_rc = design_loop(candidates_rc, analyze_frame, rc_check, E_rc)

print(f"\n第一個通過的斷面: {size_rc*100:.0f}cm")
assert size_rc == 0.20, f"預期20cm通過, 實際{size_rc}"
print("[PASS] Design Loop自動收斂結果與Case-03.6手動掃描一致(20cm)")

=== RC Design Loop ===
  嘗試 0.15: drift_util=253.8%, M_util=151.9%  -> FAIL
  嘗試 0.18: drift_util=122.4%, M_util=87.9%  -> FAIL
  嘗試 0.2: drift_util=80.3%, M_util=64.1%  -> PASS

第一個通過的斷面: 20cm
[PASS] Design Loop自動收斂結果與Case-03.6手動掃描一致(20cm)


## 第 6 課:鋼結構案例——同一套迴圈,換掉分析函式與檢核函式

證明 Design Loop 本身跟材料無關,只要傳入不同的 `build_fn`/`check_fn`
就能套用到完全不同的材料——這是 Case-03.6b 已經驗證過的可抽換架構,
這裡進一步證明連「試設迴圈」本身也可以共用同一套邏輯。

In [7]:
def analyze_frame_steel(section, E_mat):
    """section = (b_cm, t_cm), 鋼方管HSS柱; 借用analyze_frame做分析,
    需要傳入等效邊長, 使 h_eq^4/12 = 實際HSS的I """
    b_cm, t_cm = section
    Ic_cm4 = (b_cm**4 - (b_cm-2*t_cm)**4)/12
    Ic_m4 = Ic_cm4*1e-8   # cm^4 -> m^4
    h_eq = (12*Ic_m4)**0.25   # 使 h_eq^4/12 = Ic_m4, 對應analyze_frame內部算法
    return analyze_frame(h_eq, E_mat)


def steel_check(section, demand):
    b_cm, t_cm = section
    E_steel, Fy = 2100.0, 2.5   # tf/cm^2
    phi_c, phi_b, phi_v = 0.85, 0.90, 0.90

    Ag = b_cm**2 - (b_cm-2*t_cm)**2
    I = (b_cm**4 - (b_cm-2*t_cm)**4)/12
    r = (I/Ag)**0.5
    Z = (b_cm**3 - (b_cm-2*t_cm)**3)/4
    Aw = 2*(b_cm-2*t_cm)*t_cm

    lam_c = (350.0/(3.14159265*r))*(Fy/E_steel)**0.5
    Fcr = (2.71828**(-0.419*lam_c**2))*Fy if lam_c<=1.5 else (0.877/lam_c**2)*Fy
    phiPn = phi_c*Ag*Fcr*9.80665
    Mn = Fy*Z*9.80665e-4
    phiMn = phi_b*Mn
    Vn = 0.6*Fy*Aw
    phiVn = phi_v*Vn*9.80665

    p_util = demand.Pu/phiPn
    m_util = demand.Mu/phiMn
    combined = p_util+(8/9)*m_util if p_util>=0.2 else p_util/2+m_util

    drift_ok = (demand.drift < DRIFT_LIMIT) and (demand.reactions['drift2'] < DRIFT_LIMIT)
    strength_ok = combined <= 1.0

    return dict(pass_=(drift_ok and strength_ok), drift_ok=drift_ok, strength_ok=strength_ok,
                m_util=combined, drift_util=demand.drift/DRIFT_LIMIT)


candidates_steel = [(40,1.6), (40,2.5), (40,3.5), (40,5.0), (50,3.5)]

print("=== Steel Design Loop ===")
size_steel, demand_steel, result_steel, log_steel = design_loop(
    candidates_steel, analyze_frame_steel, steel_check, E_rc)

print(f"\n第一個通過的斷面: HSS{size_steel[0]:.0f}x{size_steel[0]:.0f}x{size_steel[1]:.1f}cm")
assert size_steel == (50,3.5), f"預期HSS50x50x3.5通過, 實際{size_steel}"
print("[PASS] 同一套Design Loop邏輯, 換掉分析/檢核函式後正確收斂到鋼結構的答案")

=== Steel Design Loop ===
  嘗試 (40, 1.6): drift_util=17.7%, M_util=290.8%  -> FAIL
  嘗試 (40, 2.5): drift_util=12.1%, M_util=195.0%  -> FAIL
  嘗試 (40, 3.5): drift_util=9.4%, M_util=146.8%  -> FAIL
  嘗試 (40, 5.0): drift_util=7.3%, M_util=111.3%  -> FAIL
  嘗試 (50, 3.5): drift_util=4.5%, M_util=90.6%  -> PASS

第一個通過的斷面: HSS50x50x3.5cm
[PASS] 同一套Design Loop邏輯, 換掉分析/檢核函式後正確收斂到鋼結構的答案


## 總結表

In [8]:
print("="*55)
print("Case-03.7 Demand物件與Design Loop結果總結")
print("="*55)
print(f"{'RC Design Loop結果':<26}{size_rc*100:.0f}cm")
print(f"{'  嘗試次數':<26}{len(log_rc)}")
print(f"{'Steel Design Loop結果':<26}HSS{size_steel[0]:.0f}x{size_steel[0]:.0f}x{size_steel[1]:.1f}cm")
print(f"{'  嘗試次數':<26}{len(log_steel)}")
print()
print("Case-03.7 [PASS] -- 介面穩定, 可以進到Case-04(桃園案例)")

Case-03.7 Demand物件與Design Loop結果總結
RC Design Loop結果          20cm
  嘗試次數                    3
Steel Design Loop結果       HSS50x50x3.5cm
  嘗試次數                    5

Case-03.7 [PASS] -- 介面穩定, 可以進到Case-04(桃園案例)
